In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch.nn.utils.rnn import pad_sequence
import pandas as pd
from collections import Counter
from tqdm import tqdm
import re
import os

In [2]:
# ✅ Make sure we’re inside fin_sentiment folder
os.chdir(os.path.dirname(os.path.abspath("__file__"))) if "__file__" in locals() else None

In [3]:

# Simple preprocessing
def preprocess(text):
    text = str(text).lower()
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    return text.strip()

In [4]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels, vocab=None):
        self.texts = [preprocess(t) for t in texts]
        self.labels = labels

        if vocab is None:
            words = [w for text in self.texts for w in text.split()]
            word_counts = Counter(words)
            self.vocab = {w: i+2 for i, (w, _) in enumerate(word_counts.items())}
            self.vocab["<PAD>"] = 0
            self.vocab["<UNK>"] = 1
        else:
            self.vocab = vocab

    def encode_text(self, text):
        return [self.vocab.get(w, 1) for w in text.split()]  # 1 = <UNK>

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        x = torch.tensor(self.encode_text(self.texts[idx]), dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.long)
        return x, y

def collate_fn(batch):
    texts, labels = zip(*batch)
    padded_texts = pad_sequence(texts, batch_first=True, padding_value=0)
    labels = torch.tensor(labels)
    return padded_texts, labels


In [5]:
class SentimentLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, num_classes=3):
        super(SentimentLSTM, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (h_n, _) = self.lstm(embedded)
        out = self.fc(h_n[-1])
        return out


In [6]:
# Load your CSV dataset
df = pd.read_csv("data/indian_financial_news.csv")
df.dropna(inplace=True)

In [10]:
import pandas as pd

df = pd.read_csv("data/indian_financial_news.csv")
print(df.columns)
df.head()


Index(['URL', 'Content', 'Summary', 'Sentiment'], dtype='object')


,URL,Content,Summary,Sentiment
0,https://www.moneycontrol.com/news/business/eco...,US consumer spending dropped by a record in Ap...,consumer spending plunges 13.6 percent in Apri...,Negative
1,https://www.businesstoday.in/top-story/state-r...,State-run lenders require an urgent Rs 1.2 tri...,government will have to take a bulk of the tab...,Negative
2,https://www.financialexpress.com/economy/covid...,Apparel exporters on Wednesday urged the gover...,exporters are facing issues in terms of raw ma...,Negative
3,https://www.moneycontrol.com/news/business/mar...,Asian shares battled to extend a global reboun...,the dollar loses some ground on the safe haven...,Negative
4,https://www.financialexpress.com/industry/six-...,After India’s sovereign credit rating fell to ...,six Indian public-sector undertakings have tak...,Negative


In [11]:
df = df.drop(columns=["URL"])


In [12]:
df.rename(columns={"Content": "text", "Sentiment": "label"}, inplace=True)


In [15]:
# Encode labels
le = LabelEncoder()
df["label_encoded"] = le.fit_transform(df["label"])

In [16]:
# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    df["text"].tolist(), df["label_encoded"].tolist(), test_size=0.2, random_state=42
)


In [17]:
# Create datasets and dataloaders
train_dataset = NewsDataset(X_train, y_train)
test_dataset = NewsDataset(X_test, y_test, vocab=train_dataset.vocab)

train_loader = DataLoader(train_dataset, batch_size=32, collate_fn=collate_fn, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, collate_fn=collate_fn)

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [19]:
model = SentimentLSTM(vocab_size=len(train_dataset.vocab)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for X_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/5: 100%|█████████████████████████████████████████████████████████████████████| 674/674 [54:14<00:00,  4.83s/it]


Epoch 1 | Loss: 1.0965


Epoch 2/5: 100%|█████████████████████████████████████████████████████████████████████| 674/674 [58:54<00:00,  5.24s/it]


Epoch 2 | Loss: 1.0875


Epoch 3/5: 100%|█████████████████████████████████████████████████████████████████████| 674/674 [58:20<00:00,  5.19s/it]


Epoch 3 | Loss: 1.0833


Epoch 4/5: 100%|█████████████████████████████████████████████████████████████████████| 674/674 [59:16<00:00,  5.28s/it]


Epoch 4 | Loss: 1.0792


Epoch 5/5: 100%|█████████████████████████████████████████████████████████████████████| 674/674 [54:59<00:00,  4.90s/it]

Epoch 5 | Loss: 1.0780


In [20]:
# Make sure the directory exists
os.makedirs("models", exist_ok=True)

In [21]:
torch.save({
    "model_state_dict": model.state_dict(),
    "vocab": train_dataset.vocab,
    "label_encoder": le.classes_.tolist()
}, "models/sentiment_lstm.pth")

print("✅ Model trained and saved successfully at 'models/sentiment_lstm.pth'")

✅ Model trained and saved successfully at 'models/sentiment_lstm.pth'


In [29]:
# Load back for testing
checkpoint = torch.load("models/sentiment_lstm.pth", map_location=device)
loaded_model = SentimentLSTM(vocab_size=len(checkpoint["vocab"]))
loaded_model.load_state_dict(checkpoint["model_state_dict"])
loaded_model.eval()

def predict(text):
    text = preprocess(text)
    tokens = [checkpoint["vocab"].get(w, 1) for w in text.split()]
    x = torch.tensor([tokens], dtype=torch.long).to(device)
    with torch.no_grad():
        logits = loaded_model(x)
        pred = torch.argmax(logits, dim=1).item()
        return checkpoint["label_encoder"][pred]

print(predict("Reliance Industries shares rise after new AI investment."))
print(predict("Reliance becomes a unicorn this month"))


Negative
Positive
